# Part 3 · Notebook 05 — Performance and memory

**Sessions:** S8 (Performance & memory) · S17 (NumPy for market data) · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. Measure first: time a Python loop against NumPy on a million prices.
2. Vectorize a drawdown with `np.maximum.accumulate`.
3. Compile a loop you cannot vectorize (an EMA) with `numba`.
4. Cut a DataFrame's memory with the right dtypes.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()

In [ ]:
x = p.prices_array(1_000_000)
print(f"{len(x):,} prices, {x.nbytes / 1e6:.0f} MB as float64")

## 1. Vectorizing a drawdown

$DD_t = P_t / \max_{s \le t} P_s - 1$

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
# ✍️ the drawdown of every price in one vectorized line (hint: np.maximum.accumulate)
dd = ...
dd = p.check("vectorized drawdown", dd, p.drawdown_vectorized(x))
print(f"max drawdown {dd.min():.1%}")

## 2. Compiling a loop with numba

An EMA depends on its own previous value, so it cannot be written as one NumPy expression. `numba` compiles the plain loop to machine code.

In [ ]:
def ema_py(x, alpha):
    out = np.empty(len(x))
    out[0] = x[0]
    for i in range(1, len(x)):
        out[i] = out[i - 1] + alpha * (x[i] - out[i - 1])
    return out

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from numba import njit
# ✍️ a compiled version of ema_py (hint: njit(ema_py))
ema_fast = ...
compiled = p.check("compiled with numba", hasattr(ema_fast, "signatures"), True)
if not hasattr(ema_fast, "signatures"):
    ema_fast = njit(ema_py)                       # not done yet: compile it for you so the rest runs
vals = p.check("numba EMA matches the loop", ema_fast(x[:100_000], 0.1), p.ema_loop(x[:100_000], 0.1))

In [ ]:
small = x[:200_000]
t = {"Python loop": p.timeit(ema_py, small, 0.1, repeat=2),
     "pandas ewm": p.timeit(lambda a: pd.Series(a).ewm(alpha=0.1, adjust=False).mean(), small, repeat=3),
     "numba": p.timeit(ema_fast, small, 0.1, repeat=5)}
ax = pd.Series(t).mul(1e3).plot.barh(title="EMA of 200,000 prices (ms, lower is better)", logx=True)
ax.set_xlabel("milliseconds (log scale)"); plt.show()
print({k: f"{v * 1e3:.2f} ms" for k, v in t.items()}, "— the first numba call also paid a one-off compile time")

## 3. Memory: choose dtypes

In [ ]:
rng = np.random.default_rng(0)
ticks = pd.DataFrame({"symbol": rng.choice(["SPY", "QQQ", "IWM", "TLT", "GLD"], 1_000_000),
                      "price": rng.uniform(50, 500, 1_000_000), "size": rng.integers(1, 1000, 1_000_000)})
ticks.dtypes

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
# ✍️ convert symbol to "category", price to "float32" and size to "int32" (hint: .astype({...}))
compact = ...
got = (str(getattr(compact, "dtypes", {}).get("symbol")), str(getattr(compact, "dtypes", {}).get("price")),
       str(getattr(compact, "dtypes", {}).get("size")))
got = p.check("compact dtypes", got, ("category", "float32", "int32"))
if got == ("category", "float32", "int32") and compact is Ellipsis:
    compact = ticks.astype({"symbol": "category", "price": "float32", "size": "int32"})

In [ ]:
mem = pd.DataFrame({"before (MB)": ticks.memory_usage(deep=True) / 1e6,
                    "after (MB)": compact.memory_usage(deep=True) / 1e6}).drop("Index")
mem.loc["total"] = mem.sum()
print("float32 keeps about 7 significant digits: fine for a price display, NOT for money or P&L sums.")
mem.round(1)

## Questions
1. Why measure before optimizing? Which of today's speed-ups would matter in a live loop receiving one bar a minute?
2. When is `float32` acceptable, and when is it dangerous?
3. The first numba call is slow. How would you hide that cost in a trading engine?

**Graded version:** `labs/part03/week08_advanced` (EMA) and `labs/part03/week11_data` (NumPy and pandas).